In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import torch
import timm
import numpy as np

from tqdm import tqdm
from PIL import Image

import torch.nn as nn
import torchvision.transforms as transforms

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

INPUT_PATH = os.path.join(PROJECT_PATH, "aligned_faces")

OUTPUT_PATH = os.path.join(PROJECT_PATH, "temporal_features")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Deepfakes"), exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

In [ ]:
efficientnet = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=0
)

efficientnet = efficientnet.to(device)
efficientnet.eval()

print("EfficientNet-B4 Loaded")

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
sample_video = sorted(
    os.listdir(os.path.join(INPUT_PATH, "original"))
)[0]

video_path = os.path.join(
    INPUT_PATH,
    "original",
    sample_video
)

frames = sorted(os.listdir(video_path))

print(frames)

In [ ]:
images = []

for frame in frames:
    img = cv2.imread(os.path.join(video_path, frame))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)

print(len(images))

In [ ]:
gray_images = []

for img in images:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray_images.append(gray)

print(gray_images[0].shape)

In [ ]:
diff_images = []

for i in range(len(gray_images) - 1):
    diff = cv2.absdiff(gray_images[i + 1], gray_images[i])
    diff_images.append(diff)

print(len(diff_images))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,3))

for i, img in enumerate(diff_images):
    plt.subplot(1,4,i+1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Diff {i+1}")
    plt.axis("off")

plt.show()

In [ ]:
rgb_diff_images = []

for diff in diff_images:
    rgb = cv2.cvtColor(diff, cv2.COLOR_GRAY2RGB)
    rgb_diff_images.append(rgb)

print(len(rgb_diff_images))
print(rgb_diff_images[0].shape)

In [ ]:
def extract_temporal_features(image):

    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = efficientnet(image)

    return features.squeeze().cpu().numpy()

In [ ]:
feature = extract_temporal_features(rgb_diff_images[0])

print(feature.shape)

In [ ]:
video_features = []

for img in rgb_diff_images:
    feat = extract_temporal_features(img)
    video_features.append(feat)

video_features = np.array(video_features)

print(video_features.shape)

In [ ]:
class TemporalBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1792,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        return output

In [ ]:
bilstm = TemporalBiLSTM().to(device)
bilstm.eval()

print("BiLSTM Loaded")

In [ ]:
sequence = torch.tensor(
    video_features,
    dtype=torch.float32
).unsqueeze(0).to(device)

with torch.no_grad():
    temporal_embedding = bilstm(sequence)

temporal_embedding = temporal_embedding.squeeze(0).cpu().numpy()

print(temporal_embedding.shape)

In [ ]:
save_path = os.path.join(
    OUTPUT_PATH,
    "original",
    sample_video + ".npy"
)

np.save(save_path, temporal_embedding)

print("Saved:", save_path)

In [ ]:
loaded = np.load(save_path)

print(loaded.shape)

In [ ]:
def process_video(video_folder):

    frames = sorted(os.listdir(video_folder))

    images = []

    for frame in frames:
        img = cv2.imread(os.path.join(video_folder, frame))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img)

    gray = [
        cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        for img in images
    ]

    diff = []

    for i in range(len(gray)-1):
        diff.append(cv2.absdiff(gray[i+1], gray[i]))

    rgb = [
        cv2.cvtColor(d, cv2.COLOR_GRAY2RGB)
        for d in diff
    ]

    feats = []

    for img in rgb:
        feats.append(extract_temporal_features(img))

    feats = np.array(feats)

    sequence = torch.tensor(
        feats,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = bilstm(sequence)

    return emb.squeeze(0).cpu().numpy()

In [ ]:
feature = process_video(video_path)

print(feature.shape)

In [ ]:
def process_temporal_dataset(input_root, output_root):

    videos = sorted(os.listdir(input_root))

    for video in tqdm(videos):

        video_path = os.path.join(input_root, video)

        try:
            embedding = process_video(video_path)

            np.save(
                os.path.join(output_root, video + ".npy"),
                embedding
            )

        except Exception as e:

            print(f"Skipped {video}: {e}")

In [ ]:
process_temporal_dataset(
    os.path.join(INPUT_PATH, "original"),
    os.path.join(OUTPUT_PATH, "original")
)

In [ ]:
process_temporal_dataset(
    os.path.join(INPUT_PATH, "Deepfakes"),
    os.path.join(OUTPUT_PATH, "Deepfakes")
)

In [ ]:
import os

original_files = os.listdir(os.path.join(OUTPUT_PATH, "original"))
deepfake_files = os.listdir(os.path.join(OUTPUT_PATH, "Deepfakes"))

print("Original:", len(original_files))
print("Deepfakes:", len(deepfake_files))

In [ ]:
sample = np.load(
    os.path.join(
        OUTPUT_PATH,
        "original",
        original_files[0]
    )
)

print(sample.shape)

In [ ]:
sample = np.load(
    os.path.join(
        OUTPUT_PATH,
        "Deepfakes",
        deepfake_files[0]
    )
)

print(sample.shape)